# Reading tree-ring data with dplPy

A short tour of `dplpy.readers()` and its companions. dplPy reads Tucson/ITRDB
`.rwl` files (and CSVs) into a tidy pandas `DataFrame` — one column per series,
indexed by year — and has been hardened against the many quirks found in real
ITRDB files (validated against dplR 1.7.9 across the whole ITRDB).

This notebook uses the sample files in `../tests/data/rwl/`.

In [1]:
import io, os, shutil, tempfile, contextlib, warnings
import dplpy as dpl

DATA = "../tests/data/rwl/"

def quiet(fn, *args, **kwargs):
    """Run a reader while hiding its progress printout, to keep the demo tidy."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

## 1. The basics

No arguments needed — pass a path and get back a year-indexed `DataFrame`.

In [2]:
data = dpl.readers(DATA + "ca533.rwl")
data.iloc[:5, :5]


Attempting to read input file: ca533.rwl as tucson format


SUCCESS!
File read as: tucson file

Series names:
['CAM011', 'CAM021', 'CAM031', 'CAM032', 'CAM041', 'CAM042', 'CAM051', 'CAM061', 'CAM062', 'CAM071', 'CAM072', 'CAM081', 'CAM082', 'CAM091', 'CAM092', 'CAM101', 'CAM102', 'CAM111', 'CAM112', 'CAM121', 'CAM122', 'CAM131', 'CAM132', 'CAM141', 'CAM151', 'CAM152', 'CAM161', 'CAM162', 'CAM171', 'CAM172', 'CAM181', 'CAM191', 'CAM201', 'CAM211'] 



,CAM011,CAM021,CAM031,CAM032,CAM041
Year,,,,,
626,NaN,NaN,NaN,NaN,NaN
627,NaN,NaN,NaN,NaN,NaN
628,NaN,NaN,NaN,NaN,NaN
629,NaN,NaN,NaN,NaN,NaN
630,NaN,NaN,NaN,NaN,NaN


## 2. Headers are detected automatically

Most ITRDB files begin with a 3-line metadata header. dplPy finds where the data
actually starts — no need to pass `header=True` — and tells you how many header
lines it skipped (handy for catching a rare mis-detection).

In [3]:
th = quiet(dpl.readers, DATA + "th001.rwl")
print("header lines skipped:", th.attrs["dplpy_header_lines_skipped"])
th.iloc[:3, :4]

header lines skipped: 3


/mnt/user-data/uploads/dplPy/src/dplpy/readers.py:1075: UserWarning: 1 anomalous negative value(s) (not the -9999 stop marker) were set to NaN [PATUNG@1928]
  warnings.warn(


,DOIK06,DOIK09,CNG01,CNG02
Year,,,,
1558,NaN,NaN,NaN,NaN
1559,NaN,NaN,NaN,NaN
1560,NaN,NaN,NaN,NaN


## 3. Header metadata

dplPy extracts site/sample metadata from the header. It rides along on
`df.attrs["dplpy_metadata"]`, or you can pull it directly (and cheaply, reading
only the header) with `dpl.metadata()`.

In [4]:
meta = dpl.metadata(DATA + "tx042.rwl")
meta

{'site_id': 'BSC',
 'site_name': 'Big Bend National Park incl. Camp Springs',
 'species_code': 'PSME',
 'species_name': 'Douglas Fir',
 'country_region': 'Texas',
 'elevation_m': 2057,
 'latitude': 29.25,
 'longitude': -103.3,
 'first_year': 1473,
 'last_year': 1992,
 'investigators': 'Brewster  E.Cook  N.Montagu',
 'n_header_lines': 3,
 'header_raw': ['BSC    1 Big Bend National Park incl. Camp Springs           PSME               ',
  'BSC    2 Texas        Douglas Fir       2057M  2915-10318    __    1473 1992    ',
  'BSC    3 Brewster  E.Cook  N.Montagu                                            '],
 'hemisphere_verified': True}

In [5]:
print(meta["site_id"], "|", meta["species_code"], "-", meta["species_name"],
      "|", meta["country_region"])
print("lat/lon:", meta["latitude"], meta["longitude"],
      "| hemisphere_verified:", meta["hemisphere_verified"])

BSC | PSME - Douglas Fir | Texas
lat/lon: 29.25 -103.3 | hemisphere_verified: True


The coordinate **sign** is cross-checked against the standardized country/state
in the header (e.g. a US state forces West longitude). When the region isn't a
recognized standardized name, the sign is left as-decoded and
`hemisphere_verified` is `False`, so you know it wasn't confirmed.

## 4. Flexible about the file suffix

A Tucson file needn't end in `.rwl`. For an unrecognized suffix dplPy sniffs the
content; you can also force it with `format=`.

In [6]:
tmp = tempfile.mkdtemp()
alt = os.path.join(tmp, "mydata.txt")            # a Tucson file with a .txt suffix
shutil.copy(DATA + "ca533.rwl", alt)
print("read a .txt by content sniffing:", quiet(dpl.readers, alt).shape)
print("or force it:", quiet(dpl.readers, alt, format="tucson").shape)

read a .txt by content sniffing: (1358, 34)
or force it: (1358, 34)


/mnt/user-data/uploads/dplPy/src/dplpy/readers.py:147: UserWarning: File suffix '.txt' not recognized; inferred tucson format from the file contents.
  warnings.warn("File suffix '" + FORMAT + "' not recognized; inferred "


## 5. Robust to messy files — strict mode

Real archives contain malformed files. By default (`on_error="raise"`) dplPy
refuses them with a specific, actionable message rather than silently corrupting
data.

In [7]:
for f in ["akfirmc.rwl", "viet001.rwl", "kyrg014.rwl"]:
    try:
        quiet(dpl.readers, DATA + f)
    except ValueError as e:
        detail = [ln.strip() for ln in str(e).splitlines() if ln.strip()]
        print(f"{f}:  {detail[1] if len(detail) > 1 else detail[0]}\n")

akfirmc.rwl:  Series 'FAD23B' overlaps itself at year 1210: the row beginning 1205 supplies a value for 1210, but another row begins at 1210. A row beginning at 1205 can hold only 5 value(s) before the next decade (1210), so that row appears to have one value too many, a misplaced value, or a mistyped start year. Please check that row.

viet001.rwl:  Duplicate series ID 'BDF02A': this ID is used by more than one series in the file (they overlap at year 1640) -- most often two cores mistakenly given the same code. Rename or remove the duplicate.



kyrg014.rwl:  dplPy will not guess the boundary, because reading such a series at a single precision makes part of it 10x wrong. Please split the series by precision (or correct the markers) and read it again.



## 6. Salvage mode — warn and continue

For batch processing a whole collection, `on_error="warn"` recovers as much as
possible instead of failing: it drops an unusable series (self-overlap or a
mixed-precision series), or renames a genuinely duplicated series ID, and records
every action on `df.attrs["dplpy_salvage"]`.

In [8]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d = quiet(dpl.readers, DATA + "kyrg014.rwl", on_error="warn")
print("kyrg014 salvaged ->", d.shape)
d.attrs["dplpy_salvage"]

kyrg014 salvaged -> (455, 33)


[{'series': 'kok3a',
  'issue': 'precision_shift',
  'action': 'dropped',
  'detail': 'stray -9999 near year 1898'}]

In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d2 = quiet(dpl.readers, DATA + "viet001.rwl", on_error="warn")
print("duplicate ID kept as two series:",
      [c for c in d2.columns if c.startswith("BDF02A")])
d2.attrs["dplpy_salvage"]

duplicate ID kept as two series: ['BDF02A', 'BDF02A2']


[{'series': 'BDF02A',
  'issue': 'duplicate_id',
  'action': 'renamed to BDF02A2',
  'detail': 'overlapping duplicate block kept as BDF02A2'}]

## 7. Tricky real-world cases

**Mixed measurement precision within one file** (here TMS* series measured at
0.001 mm and TWM* at 0.01 mm) and **non-ASCII series IDs** are both handled.

In [10]:
tms = quiet(dpl.readers, DATA + "TMScombined.rwl")
tms[["TMS01A", "TWM01a"]].dropna().head()

/mnt/user-data/uploads/dplPy/src/dplpy/readers.py:1075: UserWarning: 1 anomalous negative value(s) (not the -9999 stop marker) were set to NaN [TWM31a@1846]
  warnings.warn(


,TMS01A,TWM01a
Year,,
1985,2.018,0.91
1986,2.209,1.06
1987,1.897,1.37
1988,1.896,1.29
1989,1.847,1.31


In [11]:
ru = quiet(dpl.readers, DATA + "russ301.rwl")
[c for c in ru.columns if any(ord(ch) > 127 for ch in c)][:6]

['Áä89-06', 'Áä89-10', 'Áä89-12', 'Áä89-13', 'Áä89-14', 'Áä89-15']

## 8. Reading straight from a URL

`readers()` accepts an http(s) URL — for example a file from the NOAA/ITRDB
archive. (This cell needs network access; it degrades gracefully if offline.)

In [12]:
url = ("https://www.ncei.noaa.gov/pub/data/paleo/treering/"
       "measurements/northamerica/usa/ak132x.rwl")
try:
    ak = quiet(dpl.readers, url)
    print("read from URL ->", ak.shape)
except Exception as e:
    print("(network unavailable here:", type(e).__name__, "-- works on a networked machine)")

(network unavailable here: URLError -- works on a networked machine)


---
That's the tour: automatic headers, metadata with coordinate correction, flexible
formats and URLs, and a strict/salvage choice for handling the messy realities of
the ITRDB. See the docstring of `dpl.readers` for the full option list.